In [1]:
import pandas as pd
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import re

from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import Tokenizer, StopWordsRemover,StringIndexer
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.mllib.evaluation import RankingMetrics

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [4]:
meta_schema = StructType([
    StructField("main_category", StringType(), True),
    StructField("title", StringType(), True),
    StructField("average_rating", DoubleType(), True),
    StructField("rating_number", LongType(), True),
    StructField("features", ArrayType(StringType()), True),
    StructField("description", ArrayType(StringType()), True),
    StructField("price", DoubleType(), True),
    StructField("images", ArrayType(MapType(StringType(), StringType())), True),
    StructField("videos", ArrayType(MapType(StringType(), StringType())), True),
    StructField("store", StringType(), True),
    StructField("categories", ArrayType(StringType()), True),

    StructField("details", StringType(), True),

    StructField("parent_asin", StringType(), True),
    StructField("bought_together", DoubleType(), True),
    StructField("subtitle", StringType(), True),
    StructField("author", StringType(), True),
])
raw_meta_df = spark.read.text("/content/drive/MyDrive/data - MoMD/meta_Office_Products.jsonl")
meta_df = (
    raw_meta_df
    .select(from_json(col("value"), meta_schema).alias("data"))
    .select("data.*")
)

In [5]:
df_data = spark.read.format("json").option("inferSchema", "true").option("header", "true").load("/content/drive/MyDrive/data - MoMD/Office_Products.jsonl")

In [ ]:
df_data.show()

+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|      asin|helpful_vote|              images|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|
+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|B01AHHL4X2|           0|                  []| B01MZ3SD2X|   5.0|Lovely ink. Write...|1677939345945| Pretty & I love it!|AFKZENTNBQ7A7V7UX...|             true|
|B08L6H23JZ|           0|                  []| B08L6H23JZ|   4.0|Overall I’m prett...|1677939160682|2 excellent 1 ext...|AFKZENTNBQ7A7V7UX...|             true|
|B07JDZ5J46|           2|                  []| B07JDZ5J46|   1.0|[[VIDEOID:63276c1...|1660188831933|I don’t get the r...|AFKZENTNBQ7A7V7UX...|             true|
|B004MNX7EW|           0|[{IMAGE, 

In [ ]:
meta_df.show()

+--------------------+--------------------+--------------+-------------+---------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+---------------+--------------------+------+
|       main_category|               title|average_rating|rating_number|             features|         description| price|              images|              videos|               store|          categories|             details|parent_asin|bought_together|            subtitle|author|
+--------------------+--------------------+--------------+-------------+---------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+---------------+--------------------+------+
|     Office Products|Alliance Rubber 0...|           4.5|          665| [REUSABLE: These ...|[Alliance Rubber ...|  2.68|[{thumb -> https:...|[{tit

Xử lý dữ liệu review

In [6]:
meta_df_raw = meta_df
df_data_raw = df_data

In [7]:
df_data = df_data.select("asin","parent_asin","rating","timestamp","user_id","verified_purchase")

In [ ]:
df_data.printSchema()

root
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [ ]:
df_data.count()

12845712

In [ ]:
df_data = df_data.withColumnsRenamed({
    'asin':'product_id',
    'parent_asin':'product_parent',
    'rating':'review_rating',
    'timestamp':'review_date'
})

In [ ]:
user_counts = df_data.groupBy("user_id").agg(count("*").alias("number_rating"))
active_users = user_counts.filter(col("number_rating") >= 5)
active_users = active_users.drop("number_rating")
df_data = df_data.join(active_users, "user_id")

In [ ]:
product_counts = df_data.groupBy("product_id").agg(count("*").alias("number_rating"))
popular_products = product_counts.filter(col("number_rating") >= 5)
popular_products = popular_products.drop("number_rating")
df_data = df_data.join(popular_products, "product_id")

In [ ]:
df_data.count()

2165547

In [ ]:
output_path_rating = "/content/drive/MyDrive/data_parquet/RS/rating_rs.parquet"
df_data.write.mode("overwrite").parquet(output_path_rating)

Xử lý dữ liệu meta

In [ ]:
meta_df.count()

710503

In [ ]:
meta_df.printSchema()

root
 |-- main_category: string (nullable = true)
 |-- title: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- price: double (nullable = true)
 |-- parent_asin: string (nullable = true)



In [72]:
meta_df = meta_df_raw

In [8]:
meta_df = meta_df.select(
    col("parent_asin").alias("product_id"),
    "title",
    "description",
    "features",
)

In [9]:
meta_df = meta_df.dropDuplicates(["product_id"]) \
                  .withColumn("features",concat_ws(" ", col("features"))) \
                  .withColumn("description",concat_ws(" ", col("description"))) \
                  .withColumn("title",concat_ws(" ", col("title")))

In [29]:
meta_df.show(truncate=False)

+----------+-------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [30]:
total_meta = meta_df.count()

In [31]:
null_meta_counts = meta_df.select([
    sum(
        col(c).isNull().cast('int')
    ).alias(c)
    for c in meta_df.columns
])

null_meta_counts.select([
    concat(
        col(c).cast('string'),
        lit(" ("),
        round(col(c).cast('float') * 100 / total_meta, 2).cast('string'),
        lit("%)")
    ).alias(c)
    for c in meta_df.columns
]).show()

+----------+--------+-----------+--------+----------+--------+
|product_id|   title|description|features|categories|   store|
+----------+--------+-----------+--------+----------+--------+
|  0 (0.0%)|0 (0.0%)|   0 (0.0%)|0 (0.0%)|  0 (0.0%)|0 (0.0%)|
+----------+--------+-----------+--------+----------+--------+



In [32]:
chinese_pattern = r'[\u4e00-\u9fff]'

columns = meta_df.columns

for c in columns:
    count = meta_df.filter(col(c).rlike(chinese_pattern)).count()
    if count > 0:
        print(f"{c}: có {count} dòng chứa tiếng Trung")

title: có 204 dòng chứa tiếng Trung
description: có 148 dòng chứa tiếng Trung
features: có 319 dòng chứa tiếng Trung
store: có 460 dòng chứa tiếng Trung


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text) # Xóa link
    text = re.sub(r'[\u4e00-\u9fff]', ' ', text) # Xóa tiếng trung
    text = re.sub(r"[^a-zA-Z0-9\s/'.]", ' ', text) # Xóa ký tự đặc biệt trừ dấu sở hữu cách và dấu chấm
    text = re.sub(r'\s+', ' ', text) # Xóa khoảng trắng dài
    text = re.sub(r'\b\d{4,}\b', '', text) # Xóa số quá dài
    text = re.sub(r'(\d+)(oz|pack|#)', r'\1_\2', text) # gom những chữ số đi cùng đơn vị
    text = re.sub(r'\b(\d+)\s*pcs?\b', r'\1_piece', text) 
    text = re.sub(r"\.$", "", text) # Xóa dấu chấm kết thúc câu
    return text.strip()
clean_udf = udf(clean_text, StringType())

In [11]:
meta_df= meta_df.withColumn("features",clean_udf(col("features"))) \
                  .withColumn("description",clean_udf(col("description"))) \
                  .withColumn("title",clean_udf(col("title")))

In [12]:
meta_df.show(truncate=False)

+----------+-------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
output_path_meta = "/content/drive/MyDrive/data_parquet/RS/meta_rs.parquet"
# meta_df.write.mode("overwrite").parquet(output_path_meta)

In [7]:
import pandas as pd
df_meta = pd.read_parquet(output_path_meta)
df_meta

,product_id,title,description,features
0,0000306037,whitecoat clipboard lilac respiratory edition,,full size folding medical clipboard lightweigh...
1,0008529426,tolkien calendar,the official tolkien calendar this year contai...,
2,0008554404,agatha christie marple calendar,the first ever official agatha christie calend...,
3,0060515597,body for life success journal,about the author bill phillips is the founder ...,new from 1 new york times bestselling author b...
4,0061976792,the last little blue envelope 13 little blue e...,about the author maureen johnson is the bestse...,ginny blackstone thought that the biggest adve...
...,...,...,...,...
710498,B0CJ8P5DNS,2_piece folding clipboard waterproof file fold...,2 pack made of three layer foamed polypropylen...,size the size of the pp plastic clipboard is 2...
710499,B0CJRVBJVR,funny valentine's day card romantic anniversar...,funny card make your partner's anniversary a d...,funny card make your partner's anniversary a d...
710500,B0CJT326BB,tools4wisdom planner calendar 15 month dated...,,15 month planner dated october '23 to decembe...
710501,B0CKFL27BK,141a toner cartridge black 2 pack with chip co...,,with chip these 141a toner cartridges 2 pack a...


In [8]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
df_meta["clean_text"] = df_meta["title"] + " " + df_meta["features"] + " " + df_meta["description"]

In [18]:
embeddings = model.encode(
    df_meta["clean_text"].tolist(),
    show_progress_bar=True,
    batch_size=64
)
# tràn Ram

Batches:   0%|          | 0/11102 [00:00<?, ?it/s]

In [23]:
import pyarrow as pa
import pyarrow.parquet as pq

batch_size = 50000
writer = None

for start in range(0, len(df_meta), batch_size):
    end = start + batch_size
    batch = df_meta.iloc[start:end].copy()

    embeddings = model.encode(
        batch["clean_text"].tolist(),
        batch_size=64,
        show_progress_bar=True
    )

    batch["embedding"] = embeddings.tolist()

    table = pa.Table.from_pandas(batch[["product_id", "embedding"]])

    if writer is None:
        writer = pq.ParquetWriter("item_embedding.parquet", table.schema)

    writer.write_table(table)

    print(f"Saved batch {start} -> {end}")

if writer:
    writer.close()

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Saved batch 0 -> 50000


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Saved batch 50000 -> 100000


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

...

Saved batch 600000 -> 650000


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Saved batch 650000 -> 700000


Batches:   0%|          | 0/165 [00:00<?, ?it/s]

Saved batch 700000 -> 750000


In [24]:
df_embedding = pd.read_parquet("item_embedding.parquet")

In [26]:
pd.set_option('display.max_colwidth', None)

In [28]:
df_embedding.head()

,product_id,embedding
0,0000306037,"[-0.12555302679538727, 0.09906124323606491, -0.017297780141234398, -0.024684220552444458, -0.02021508663892746, -0.05026417225599289, 0.0727296769618988, 0.06389782577753067, 0.017076391726732254, -0.00015298384823836386, -0.0007342112367041409, 0.08739827573299408, -0.002871930366382003, 0.03246589004993439, -0.07278109341859818, 0.02876361459493637, 0.009181790053844452, -0.02852225862443447, 0.029881885275244713, 0.09569589048624039, 0.0163209717720747, 0.1343345046043396, 0.0061849928461015224, 0.01899814046919346, -0.09267666935920715, 0.07332170009613037, -0.04207342490553856, -0.032755251973867416, 0.057356659322977066, -0.0038867180701345205, 0.09527496993541718, -0.060249410569667816, 0.06423840671777725, 0.02871454320847988, 0.00624098489060998, -0.04811490699648857, 0.018542079254984856, 0.0402732715010643, -0.031183619052171707, 0.010860511101782322, -0.05822526291012764, -0.04071443900465965, -0.05931970849633217, 0.057768139988183975, -0.012367923744022846, 0.026566438376903534, -0.020418480038642883, 0.008516835980117321, 0.01368502713739872, 0.04151761159300804, -0.00612029479816556, -0.07846900075674057, -0.12176377326250076, 0.017172837629914284, -0.03303269296884537, -0.04602867364883423, -0.06591469794511795, -0.033583756536245346, -0.05176196247339249, -0.05358988791704178, -0.06718026101589203, 0.029689984396100044, 0.02193906530737877, 0.045956701040267944, 0.022216083481907845, 0.09513342380523682, -0.03188088908791542, -0.01319093070924282, 0.022253137081861496, -0.03237836807966232, -0.04669223725795746, -0.053393881767988205, 0.01328516099601984, 0.09807851165533066, -0.02927984483540058, 0.02080625481903553, 0.038605380803346634, -0.05249982699751854, 0.008794637396931648, -0.023848069831728935, 0.00816244725137949, -0.019309906288981438, 0.03839047998189926, 0.10271231085062027, 0.005131259560585022, 0.006280186586081982, -0.04158413037657738, 0.0007975035696290433, -0.031870000064373016, -0.05805813893675804, 0.0817800834774971, 0.023890241980552673, 0.05999733507633209, 0.017850108444690704, -0.03926476091146469, -0.009555453434586525, -0.05540744960308075, 0.03298250585794449, 0.0044542583636939526, 0.036864105612039566, ...]"
1,0008529426,"[-0.004104066174477339, -0.019372206181287766, 0.030008632689714432, -0.025627896189689636, 0.0734945684671402, -0.012533874250948429, -0.1428128331899643, -0.01496238261461258, 0.012759395875036716, -0.020985914394259453, -0.0994969978928566, -0.0223526693880558, 0.032245934009552, -0.03654516860842705, 0.0012051602825522423, 0.002953690243884921, -0.07397463172674179, -0.020135648548603058, 0.033934611827135086, -0.07294750958681107, 0.004179357085376978, 0.020791534334421158, -0.0036205884534865618, 0.036664485931396484, 0.04284307360649109, 0.005140196532011032, 0.03963885456323624, -0.0504218153655529, -0.014560461975634098, 0.0673854872584343, -0.0076028723269701, 0.08753211796283722, -0.025127587839961052, -0.024024875834584236, -0.009188895113766193, 0.09972129017114639, -0.10641554743051529, 0.01412620022892952, 0.002991184126585722, -0.07339416444301605, -0.023318763822317123, -0.05658077448606491, 0.04018841311335564, -0.0645633265376091, -0.006079362239688635, -0.0007945604156702757, -0.07878988236188889, 0.03337268531322479, -0.041728485375642776, 0.07915887981653214, 0.029119368642568588, -0.07614865154027939, -0.03931021690368652, -0.040560413151979446, 0.10766872018575668, -0.007918953895568848, -0.1288386583328247, -0.09349136054515839, 0.051570624113082886, 0.026260213926434517, -0.05980580300092697, 0.022648675367236137, -0.04820971563458443, 0.014039985835552216, -0.03430285304784775, 0.05124075710773468, 0.062472887337207794, 0.002061366569250822, 0.014891651459038258, -0.11028030514717102, 0.0057808286510407925, -0.010913630947470665, -0.007321578450500965, -0.0745648592710495, 0.04448802396655083, 0.042412251234054565, -0.11121100187301636, 0.007698793895542622, -0.043278735131025314, -0.1175775676965